# Stroke Prediction: Exploratory Analysis, Clustering & Classification

**Author:** Sebastian Silva  
**Dataset:** [Healthcare Dataset — Stroke Data (Kaggle)](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)

---

## Overview

This project investigates stroke prediction using a real-world healthcare dataset of 5,110 patients.  
The analysis is structured in three phases:

1. **Baseline comparison** — Logistic Regression with and without SMOTE, evaluated using Deviance R² and standard classification metrics to understand the effect of oversampling on an imbalanced dataset.  
2. **Unsupervised clustering** — Four clustering algorithms applied to explore natural patient groupings, evaluated by silhouette score and visualized via PCA.  
3. **Full pipeline comparison** — Logistic Regression, Random Forest, and a Neural Network trained on all features using a preprocessing + SMOTE pipeline, with model selection justified by recall — the clinically relevant metric for stroke detection.

**Key challenge:** Stroke cases represent only ~4.9% of observations. Standard accuracy is a misleading metric in this context — a model predicting "no stroke" for every patient would achieve 95% accuracy while catching zero actual cases. All modelling decisions are made with this imbalance in mind.

> **To run:** Update the file path in the data loading cell to match your local environment.


## 1. Imports

In [ ]:
# All imports consolidated here
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans, MeanShift
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, auc, silhouette_score
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

## 2. Data Loading & Preprocessing

In [ ]:
# ── Update this path to point to your local copy of the dataset ──
DATA_PATH = "healthcare-dataset-stroke-data.csv"

data = pd.read_csv(DATA_PATH)

# Drop non-informative ID column
data.drop('id', axis=1, inplace=True)

data.describe()

               age  hypertension  heart_disease  avg_glucose_level  \
count  5110.000000   5110.000000    5110.000000        5110.000000   
mean     43.226614      0.097456       0.054012         106.147677   
std      22.612647      0.296607       0.226063          45.283560   
min       0.080000      0.000000       0.000000          55.120000   
25%      25.000000      0.000000       0.000000          77.245000   
50%      45.000000      0.000000       0.000000          91.885000   
75%      61.000000      0.000000       0.000000         114.090000   
max      82.000000      1.000000       1.000000         291.050000   

In [ ]:
# Check for missing values
data.isnull().sum()

gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

### Missing Data

There are **201 missing values** in the `bmi` column (~3.9% of rows).  
Because this is medical data, dropping these patients would risk introducing selection bias — patients with missing BMI may differ systematically from those with complete records.  
**Median imputation** is used to preserve all observations while minimising distortion of the BMI distribution.

In [ ]:
# Median imputation — avoids distorting the BMI distribution
data['bmi'] = data['bmi'].fillna(data['bmi'].median())

# Confirm no remaining missing values
data.isnull().sum()

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

---

## 3. Exploratory Data Analysis

The goal of EDA is to understand the distributions of key features and their relationship to stroke outcome before any modelling takes place.

In [ ]:
# Summary statistics split by stroke outcome
numeric_features = ['age', 'avg_glucose_level', 'bmi']

data.groupby('stroke')[numeric_features].agg(['mean', 'median', 'min', 'max', 'std'])

              age                               avg_glucose_level          \
             mean median   min   max        std              mean  median   
stroke                                                                      
0       41.971545   43.0  0.08  82.0  22.291940        104.795513   91.47   
1       67.728193   71.0  1.32  82.0  12.727419        132.544739  105.22   

                                        bmi                               
          min     max        std       mean median   min   max       std  
stroke                                                          
0        55.12  291.05  45.748060  28.824438   28.1  10.3  97.6   7.804  
1        57.96  267.08  53.257413  30.468263   28.1  14.5  97.6   9.064  

In [ ]:
# Pairplot — overall visual separation between stroke classes
sns.pairplot(data, hue='stroke')
plt.suptitle("Pairplot by Stroke Outcome", y=1.02)
plt.show()

In [ ]:
# Stroke count by gender
sns.countplot(x='gender', hue='stroke', data=data)
plt.title('Stroke Count by Gender')
plt.show()

In [ ]:
# Stroke count by smoking status
sns.countplot(x='smoking_status', hue='stroke', data=data)
plt.title('Stroke Count by Smoking Status')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: Age vs BMI and Age vs Glucose, with regression line and R²
numeric_vars = ['bmi', 'avg_glucose_level']
fig, axes = plt.subplots(1, len(numeric_vars), figsize=(12, 4))

for i, var in enumerate(numeric_vars):
    r = data['age'].corr(data[var])
    r2 = r ** 2

    sns.scatterplot(x='age', y=var, data=data, alpha=0.6, ax=axes[i])
    sns.regplot(x='age', y=var, data=data, scatter=False, color='red', ax=axes[i])
    axes[i].set_title(f'Age vs {var}  (R² = {r2:.3f})')
    axes[i].set_xlabel('Age')
    axes[i].set_ylabel(var)

plt.tight_layout()
plt.show()

In [ ]:
# Violin plot — age distribution by stroke outcome
sns.violinplot(x='stroke', y='age', data=data)
plt.title('Age Distribution by Stroke Outcome')
plt.show()

In [ ]:
# Box plots — age, glucose, BMI by stroke outcome
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = ['skyblue', 'salmon']

for i, feature in enumerate(numeric_features):
    sns.boxplot(
        x='stroke', y=feature, hue='stroke',
        data=data, palette=palette, ax=axes[i], legend=False
    )
    axes[i].set_title(f'{feature} vs Stroke')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — numerical features only
plt.figure(figsize=(8, 6))
sns.heatmap(data[numeric_features].corr(), annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

In [ ]:
# Pearson correlation between each numeric feature and stroke
for feature in numeric_features:
    corr, p_value = pearsonr(data[feature], data['stroke'])
    print(f"Pearson's r — {feature} vs stroke: {corr:.2f}  (p = {p_value:.4f})")

Pearson's r — age vs stroke: 0.25  (p = 0.0000)
Pearson's r — avg_glucose_level vs stroke: 0.13  (p = 0.0000)
Pearson's r — bmi vs stroke: 0.04  (p = 0.0098)


### EDA Summary

- **Age** shows the strongest association with stroke (r = 0.25, p < 0.001). Stroke patients have a median age of 71 vs. 43 for non-stroke patients — a striking separation.
- **Average glucose level** shows a moderate association (r = 0.13). Stroke patients have a higher median glucose (105 vs. 91), and the upper quartile extends much further (196 vs. 113), suggesting elevated glucose is a risk marker at the extremes.
- **BMI** shows a weak and less consistent association (r = 0.04). Median BMI is nearly identical across stroke and non-stroke groups, indicating it contributes less as a standalone predictor.
- The pairplot confirms limited linear separation between classes — stroke risk is **multifactorial**, which motivates including all features in the final models rather than filtering by correlation alone.
- The most correlated numeric pair is **age and BMI** (r = 0.32), followed by **age and average glucose level** (r = 0.24), both modest enough to avoid serious multicollinearity concerns.

---

## 4. Phase 1 — Logistic Regression: Baseline vs. SMOTE

### Motivation

The dataset is heavily imbalanced: stroke cases account for only ~4.9% of observations. A naive model will simply learn to predict the majority class, achieving high accuracy while detecting zero strokes.

**SMOTE** (Synthetic Minority Oversampling Technique) addresses this by generating synthetic minority-class observations in feature space during training only, preventing data leakage into the test set.

**Deviance R²** (McFadden's pseudo-R²) is used here instead of standard R² because the outcome is binary. It measures how much better the fitted model is than a null model that always predicts the base rate:

$$R^2_{\text{dev}} = 1 - \frac{LL_{\text{model}}}{LL_{\text{null}}}$$

A value of 0 means the model adds nothing; 1 is a perfect fit. **Negative values indicate the model fits worse than the null** — which, as shown below, is exactly what happens when SMOTE miscalibrates predicted probabilities.

In [ ]:
# ── Stratified train/test split (numeric features only for Phase 1 comparison) ──
features_phase1 = ['age', 'avg_glucose_level', 'bmi']
X_p1 = data[features_phase1]
y_p1 = data['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X_p1, y_p1, test_size=0.2, random_state=42, stratify=y_p1
)

print("Training class distribution:")
print(y_train.value_counts())
print(f"\nStroke prevalence in training set: {y_train.mean():.1%}")

Training class distribution:
stroke
0    3889
1     199
Name: count, dtype: int64

Stroke prevalence in training set: 4.9%


In [ ]:
# Per-feature Deviance R² — which individual predictor explains the most variance?
eps = 1e-15
deviance_r2_results = {}

for feature in features_phase1:
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train[[feature]], y_train)
    y_prob = model.predict_proba(X_test[[feature]])[:, 1]

    LL_model = np.sum(y_test * np.log(y_prob + eps) + (1 - y_test) * np.log(1 - y_prob + eps))
    p_null = np.mean(y_test)
    LL_null = np.sum(y_test * np.log(p_null + eps) + (1 - y_test) * np.log(1 - p_null + eps))

    deviance_r2_results[feature] = 1 - (LL_model / LL_null)

pd.DataFrame.from_dict(deviance_r2_results, orient='index', columns=['Deviance R²'])  .sort_values('Deviance R²', ascending=False)

                   Deviance R²
age                   0.194639
avg_glucose_level     0.048858
bmi                   0.003127

In [ ]:
# ── Baseline Logistic Regression (no SMOTE) ──
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_prob = log_model.predict_proba(X_test)[:, 1]

LL_model = np.sum(y_test * np.log(y_prob + eps) + (1 - y_test) * np.log(1 - y_prob + eps))
p_null   = np.mean(y_test)
LL_null  = np.sum(y_test * np.log(p_null + eps) + (1 - y_test) * np.log(1 - p_null + eps))
deviance_r2 = 1 - (LL_model / LL_null)

# ── Logistic Regression + SMOTE ──
smote = SMOTE(sampling_strategy=0.10, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE:")
print(y_train_sm.value_counts())
print(f"Stroke prevalence after SMOTE: {y_train_sm.mean():.1%}")

log_model_sm = LogisticRegression(max_iter=1000)
log_model_sm.fit(X_train_sm, y_train_sm)
y_prob_sm = log_model_sm.predict_proba(X_test)[:, 1]

LL_model_sm  = np.sum(y_test * np.log(y_prob_sm + eps) + (1 - y_test) * np.log(1 - y_prob_sm + eps))
deviance_r2_sm = 1 - (LL_model_sm / LL_null)

Class distribution after SMOTE:
stroke
0    3889
1     388
Name: count, dtype: int64
Stroke prevalence after SMOTE: 9.1%


In [ ]:
# Predictions
y_pred    = log_model.predict(X_test)
y_pred_sm = log_model_sm.predict(X_test)

# Metrics comparison table
metrics_phase1 = pd.DataFrame({
    "Model":        ["Logistic", "Logistic + SMOTE"],
    "Accuracy":     [accuracy_score(y_test, y_pred),    accuracy_score(y_test, y_pred_sm)],
    "Precision":    [precision_score(y_test, y_pred, zero_division=0), precision_score(y_test, y_pred_sm, zero_division=0)],
    "Recall":       [recall_score(y_test, y_pred, zero_division=0),    recall_score(y_test, y_pred_sm, zero_division=0)],
    "F1 Score":     [f1_score(y_test, y_pred, zero_division=0),        f1_score(y_test, y_pred_sm, zero_division=0)],
    "ROC AUC":      [roc_auc_score(y_test, y_prob),     roc_auc_score(y_test, y_prob_sm)],
    "Deviance R²":  [deviance_r2, deviance_r2_sm]
}).round(3)

metrics_phase1

              Model  Accuracy  Precision  Recall  F1 Score  ROC AUC  Deviance R²
0          Logistic     0.951      0.000    0.00     0.000    0.840        0.204
1  Logistic + SMOTE     0.749      0.137    0.78     0.233    0.840       -1.522

In [ ]:
# Confusion matrices — side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred_plot, title in zip(
    axes,
    [y_pred, y_pred_sm],
    ["Logistic (no SMOTE)", "Logistic + SMOTE"]
):
    cm = confusion_matrix(y_test, y_pred_plot)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Pred: No Stroke', 'Pred: Stroke'],
        yticklabels=['Actual: No Stroke', 'Actual: Stroke']
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()

### Phase 1 Findings

**Without SMOTE**, the model achieves 95.1% accuracy but predicts zero stroke cases — it has effectively learned to always output the majority class. Recall = 0.00.

**With SMOTE**, the model catches 78% of actual stroke cases (recall = 0.78), at the cost of accuracy dropping to 74.9% due to increased false positives.

**The Deviance R² of −1.52 for the SMOTE model is the most important result in this phase.** A negative value means the SMOTE-trained model produces *worse* probability estimates than simply predicting the base rate for every patient. This is not a failure of SMOTE itself — it is a calibration problem: SMOTE inflates the minority class to 9% of training data, but the true prevalence in the test set is still ~5%. The model's learned probabilities reflect the synthetic training distribution, not the real one, which collapses the log-likelihood.

**The key takeaway:** SMOTE improves *detection* (recall) but degrades *calibration* (probability estimates). In a clinical setting, this trade-off must be weighed against the cost of missed strokes vs. false alarms. For this analysis, maximising recall is prioritised.

---

## 5. Phase 2 — Unsupervised Clustering

### Motivation

Before building multi-feature classifiers, unsupervised clustering is applied to explore whether the patient data contains natural groupings — and whether those groupings align with stroke risk. This is a purely exploratory step: no labels are used during clustering.

All features are preprocessed consistently (StandardScaler for numeric, OneHotEncoder for categorical) before clustering, identical to the preprocessing used in Phase 3.

In [ ]:
# ── Preprocessing ──
X_all = data.drop('stroke', axis=1)
y_all = data['stroke']

categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
numeric_cols     = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

X_processed = preprocessor.fit_transform(X_all)
print(f"Preprocessed feature matrix shape: {X_processed.shape}")

Preprocessed feature matrix shape: (5110, 16)


In [ ]:
# ── Run four clustering algorithms ──
agg = AgglomerativeClustering(n_clusters=3)
labels_agg = agg.fit_predict(X_processed)
silhouette_agg = silhouette_score(X_processed, labels_agg)

kmeans = KMeans(n_clusters=3, random_state=42)
labels_kmeans = kmeans.fit_predict(X_processed)
silhouette_kmeans = silhouette_score(X_processed, labels_kmeans)

mbk = MiniBatchKMeans(n_clusters=3, random_state=42, batch_size=3584)
labels_mbk = mbk.fit_predict(X_processed)
silhouette_mbk = silhouette_score(X_processed, labels_mbk)

ms = MeanShift(bandwidth=3)
labels_ms = ms.fit_predict(X_processed)
silhouette_ms = silhouette_score(X_processed, labels_ms)

cluster_results = pd.DataFrame({
    'Method':           ['Agglomerative', 'K-Means', 'MiniBatch K-Means', 'Mean Shift'],
    'Silhouette Score': [silhouette_agg, silhouette_kmeans, silhouette_mbk, silhouette_ms]
}).round(3)

cluster_results

              Method  Silhouette Score
0      Agglomerative             0.389
1            K-Means             0.193
2  MiniBatch K-Means             0.196
3         Mean Shift             0.385

In [ ]:
# ── Stroke rate by cluster — does cluster membership align with stroke risk? ──
data_with_clusters = data.copy()
data_with_clusters['cluster_agg'] = labels_agg
data_with_clusters['cluster_ms']  = labels_ms

print("Stroke rate by Agglomerative cluster:")
print(data_with_clusters.groupby('cluster_agg')['stroke'].agg(['mean', 'sum', 'count']).round(3))

print("\nStroke rate by Mean Shift cluster:")
print(data_with_clusters.groupby('cluster_ms')['stroke'].agg(['mean', 'sum', 'count']).round(3))

Stroke rate by Agglomerative cluster:
             mean  sum  count
cluster_agg                  
0           0.021   28   1317
1           0.082  143   1749
2           0.038   78   2044

Stroke rate by Mean Shift cluster:
            mean  sum  count
cluster_ms                  
0          0.047  234   4959
1          0.093   13    140
2          0.200    2     10

In [ ]:
# Cluster demographic profile — Agglomerative (best silhouette, most interpretable)
profile_cols = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']
print("Mean feature values by Agglomerative cluster:")
print(data_with_clusters.groupby('cluster_agg')[profile_cols].mean().round(2))

In [ ]:
# ── PCA for 3D cluster visualisation ──
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_processed)

print(f"Variance explained by PC1, PC2, PC3: "
      f"{pca.explained_variance_ratio_.cumsum()[-1]:.1%} total")

feature_names = preprocessor.get_feature_names_out()

loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2', 'PC3'],
    index=feature_names
)

print("\nTop 5 features driving PC1 (general health & age axis):")
print(loadings['PC1'].abs().sort_values(ascending=False).head(5))

In [ ]:
# 3D scatter — all four clustering methods
cluster_methods = {
    'Agglomerative':   labels_agg,
    'K-Means':         labels_kmeans,
    'MiniBatch K-Means': labels_mbk,
    'Mean Shift':      labels_ms
}

fig = plt.figure(figsize=(14, 10))

for i, (name, labels) in enumerate(cluster_methods.items(), 1):
    ax = fig.add_subplot(2, 2, i, projection='3d')
    ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
               c=labels, cmap='viridis', alpha=0.6)
    ax.set_title(name)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')

plt.tight_layout()
plt.show()

### Phase 2 Findings

**Silhouette scores:** Agglomerative (0.389) and Mean Shift (0.385) produced more coherent cluster structures than K-Means (0.193) or MiniBatch K-Means (0.196). Scores below ~0.5 indicate overlapping clusters, which is expected given the complexity and continuous nature of health data.

**Clinical interpretation of Agglomerative clusters:**
- **Cluster 0** (stroke rate: 2.1%) — younger patients with lower glucose and BMI; the lowest-risk group
- **Cluster 1** (stroke rate: 8.2%) — the highest-risk group, likely older with more comorbidities
- **Cluster 2** (stroke rate: 3.8%) — intermediate risk profile

The 4× difference in stroke rate between Cluster 0 and Cluster 1 confirms that the clusters are not arbitrary — they capture meaningful variation in patient risk profiles. PC1 is dominated by age, BMI, hypertension, and glucose level, which aligns with established clinical risk factors for stroke.

The PCA visualisations show that Agglomerative and Mean Shift produce more spatially separated clusters, consistent with their higher silhouette scores. K-Means and MiniBatch K-Means show heavier overlap.

---

## 6. Phase 3 — Full Pipeline Model Comparison

### Motivation

Phase 1 used only three numeric features to isolate the effect of SMOTE. Phase 3 uses the **full feature set** (all 10 variables, 16 after encoding) with a proper preprocessing pipeline to compare three model families:

- **Logistic Regression** — linear baseline, interpretable
- **Random Forest** — non-linear ensemble, handles interactions well
- **Neural Network (MLP)** — flexible non-linear model

All three pipelines apply identical preprocessing (StandardScaler + OneHotEncoder) and SMOTE inside the pipeline to prevent data leakage. SMOTE is applied only to the training fold.

In [ ]:
# ── Train/test split on full feature set ──
X_full = data.drop('stroke', axis=1)
y_full = data['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print(f"Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")
print(f"Stroke prevalence — train: {y_train.mean():.1%} | test: {y_test.mean():.1%}")

Training set: 4088 rows | Test set: 1022 rows
Stroke prevalence — train: 4.9% | test: 4.9%


In [ ]:
# ── Logistic Regression pipeline ──
lr_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
print("Logistic Regression pipeline fitted.")

Logistic Regression pipeline fitted.


In [ ]:
# ── Random Forest pipeline ──
rf_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]
print("Random Forest pipeline fitted.")

Random Forest pipeline fitted.


In [ ]:
# ── Neural Network pipeline ──
nn_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42))
])

nn_pipeline.fit(X_train, y_train)
y_pred_nn = nn_pipeline.predict(X_test)
y_prob_nn = nn_pipeline.predict_proba(X_test)[:, 1]
print("Neural Network pipeline fitted.")

Neural Network pipeline fitted.


In [ ]:
# ── Metrics comparison ──
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Random Forest':       (y_pred_rf, y_prob_rf),
    'Neural Network':      (y_pred_nn, y_prob_nn)
}

results = []
for name, (preds, probs) in models.items():
    results.append({
        'Model':         name,
        'Accuracy':      accuracy_score(y_test, preds),
        'Precision':     precision_score(y_test, preds, zero_division=0),
        'Recall':        recall_score(y_test, preds, zero_division=0),
        'F1 Score':      f1_score(y_test, preds, zero_division=0),
        'ROC AUC':       roc_auc_score(y_test, probs)
    })

results_df = pd.DataFrame(results).round(3)
results_df

                 Model  Accuracy  Precision  Recall  F1 Score  ROC AUC
0  Logistic Regression     0.750      0.140    0.80     0.239    0.844
1        Random Forest     0.932      0.167    0.10     0.125    0.756
2       Neural Network     0.886      0.132    0.24     0.170    0.761

In [ ]:
# ── Confusion matrices — all three models ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, (preds, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Pred: No Stroke', 'Pred: Stroke'],
        yticklabels=['Actual: No Stroke', 'Actual: Stroke']
    )
    # Annotate with recall
    recall = recall_score(y_test, preds, zero_division=0)
    ax.set_title(f'{name}\nRecall = {recall:.2f}')

plt.tight_layout()
plt.show()

In [ ]:
# ── Grouped bar chart — metric comparison ──
metrics_melted = results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_melted, x='Metric', y='Score', hue='Model')
plt.title('Classification Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(title='Model')
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ──
plt.figure(figsize=(8, 6))

for name, (_, probs) in models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.show()

### Phase 3 Findings

| Model | Accuracy | Recall | ROC AUC | Assessment |
|---|---|---|---|---|
| Logistic Regression | 0.750 | **0.80** | **0.844** | Best for stroke detection |
| Random Forest | 0.932 | 0.10 | 0.756 | High accuracy, misses 90% of strokes |
| Neural Network | 0.886 | 0.24 | 0.761 | Intermediate; underperforms on recall |

**Random Forest's 93.2% accuracy is misleading.** The confusion matrix reveals it catches only ~5 of the 50 actual stroke cases in the test set — it has largely defaulted to the majority class despite SMOTE. Its ROC AUC of 0.756 is also the lowest of the three, confirming weak discrimination.

**Logistic Regression is the strongest model for this problem.** It detects 80% of stroke cases (catching ~40 of 50 in the test set), with the highest ROC AUC (0.844). The precision of 14% means it also generates false positives, but in a medical screening context — where a missed stroke is far more costly than an unnecessary follow-up — high recall is the correct objective.

**The Neural Network underperforms for this dataset size.** With only ~200 positive training cases (even after SMOTE), there is insufficient data to take advantage of the MLP's capacity. Logistic Regression is better suited to small, tabular, imbalanced datasets.

---

## 7. Conclusion

This project evaluated stroke prediction across three analytical phases using a highly imbalanced healthcare dataset (~5% stroke prevalence).

**Phase 1** demonstrated that SMOTE dramatically improves recall (0% → 78%) but at the cost of probability calibration — a negative Deviance R² reveals that the SMOTE-trained model's predicted probabilities are worse than simply using the base rate. This is a known limitation when SMOTE alters the training class prior without correcting the decision threshold.

**Phase 2** found that Agglomerative Clustering and Mean Shift produce more coherent patient groupings (silhouette ≈ 0.39) than K-Means variants (≈ 0.19). The best-performing clusters capture a clinically meaningful 4× difference in stroke rate (2.1% vs. 8.2%), with PC1 dominated by age, BMI, hypertension, and glucose — consistent with established stroke risk factors.

**Phase 3** confirmed that Logistic Regression is the best-performing model for stroke detection on this dataset (Recall = 0.80, ROC AUC = 0.844). Random Forest achieves misleadingly high accuracy (93.2%) while catching only 10% of strokes. The Neural Network offers no advantage given the limited minority class sample size.

### Limitations

- A **single 80/20 split** is used throughout. Cross-validation (e.g., stratified k-fold) would produce more reliable estimates given the small number of positive cases.
- **SMOTE's sampling strategy** (10%) was selected based on reasonable judgment but was not formally optimised via grid search.
- **No temporal or external validation** — the model's generalisability to other populations or time periods is unknown.
- **Pearson correlation** was used for feature-outcome association. For categorical features, Cramér's V or chi-square tests would be more appropriate.

### Future Work

- Threshold tuning: optimise the classification threshold for recall vs. precision trade-off using the precision-recall curve
- Cross-validation: implement stratified k-fold to stabilise metric estimates
- Feature engineering: explore interaction terms (e.g., age × hypertension)

